# 02 — Merging 4-digit ISCO codes (SHP add-on) with the ILO exposure index

**Rule:** this notebook never displays individual SHP rows (no `head()`, no `display(df)`).
The FORS contract forbids making the data accessible to third parties, including AI tools.
Only aggregate information is displayed: `info()`, `shape`, `value_counts()`, means.

**Before running:**
- have your own access to the FORS add-on (ref. 3260) and unzip `Data_with_additional_contract.zip` in your data folder;
- have run `01_shp_cleaning.py`, with its output saved to `Data/private/shp_population_income.parquet`.

In [ ]:
import pandas as pd
import numpy as np

## 1. Paths
Relative paths from the `Notebooks/` folder, as in the course exercises (`./data/...`).

In [ ]:
ADDON_PATH = "../Data/raw/SHP/swissubase_3260_1_1/Data_with_additional_contract/SHP_EXT_ISCO4_OCCUPA_USER.dta"
SHP_PATH   = "../Data/private/shp_population_income.parquet"   # output of 01_shp_cleaning
ILO_PATH   = "../Data/processed/ILO_exposure_by_occupation_ISCO4.csv"
OUT_PATH   = "../Data/private/shp_isco4_ilo.parquet"

## 2. Inspect the add-on (no rows displayed)
The file is in **wide format**: one row per person, one column per wave.
`IS4MAJyy` = 4-digit ISCO-08 code of the **main job** in year 20yy.

In [ ]:
addon = pd.read_stata(ADDON_PATH, convert_categoricals=False)
print(addon.shape)
addon.info()

## 3. Reshape to long format (`melt`, Week 2)
One row per person **and** year, as in the cleaned file.

In [ ]:
isco4_cols = ["IS4MAJ16", "IS4MAJ17", "IS4MAJ18", "IS4MAJ19", "IS4MAJ20",
              "IS4MAJ21", "IS4MAJ22", "IS4MAJ23", "IS4MAJ24"]

addon = addon.melt(id_vars="idpers", value_vars=isco4_cols,
                   var_name="variable", value_name="is4maj")

# The year comes from the last 2 characters of the column name
addon["year"] = 2000 + addon["variable"].str[-2:].astype(int)
addon = addon.drop("variable", axis=1)

print(addon.shape)
print(addon["year"].value_counts().sort_index())

## 4. Clean the 4-digit ISCO code
**Non-responses are coded as negative numbers** (-1, -2, -3, -8) → `NaN` (Week 2, missing values encoded by specific values).
The code is converted to a **4-character string** (`zfill`) to keep the leading zero, like `isco3` in the cleaning script.

In [ ]:
addon.loc[addon["is4maj"] < 0, "is4maj"] = np.nan
addon["isco4"] = addon["is4maj"].apply(lambda x: str(int(x)).zfill(4) if pd.notna(x) else np.nan)
addon = addon[["idpers", "year", "isco4"]]

print("Duplicates idpers + year:", addon.duplicated(subset=["idpers", "year"]).sum())
print("Share of missing ISCO 4:", round(addon["isco4"].isna().mean(), 3))
print("Number of distinct codes:", addon["isco4"].nunique())

## 5. Merge with the cleaned file
`how="left"` keeps every row of the cleaned file; `indicator=True` (Week 4) shows whether each row found a code.

In [ ]:
shp = pd.read_parquet(SHP_PATH)
print("Rows before merge:", shp.shape[0])

df = pd.merge(shp, addon, on=["idpers", "year"], how="left", indicator=True)
print("Rows after merge:", df.shape[0])   # must be identical
print(df["_merge"].value_counts())
df = df.drop("_merge", axis=1)

# Check: the first 3 digits of isco4 must match isco3
both = df[df["isco4"].notna() & df["isco3"].notna()]
print("Consistent share:", round((both["isco4"].str[:3] == both["isco3"]).mean(), 3))

## 6. Merge with the 4-digit ILO index

In [ ]:
ilo = pd.read_csv(ILO_PATH)
ilo["isco4"] = ilo["ISCO_08"].astype(str).str.zfill(4)

ilo4 = ilo[["isco4", "exposure_mean_2025", "exposure_category_2025"]]
ilo4 = ilo4.rename(columns={"exposure_mean_2025": "exposure4",
                            "exposure_category_2025": "exposure4_cat"})
print("Duplicates in the ILO file:", ilo4.duplicated(subset=["isco4"]).sum())   # must be 0

df = pd.merge(df, ilo4, on="isco4", how="left", indicator=True)
print("Rows after merge:", df.shape[0])
print(df["_merge"].value_counts())

# SHP codes without a score (codes and number of observations only)
no_score = df[(df["isco4"].notna()) & (df["_merge"] == "left_only")]
print(no_score["isco4"].value_counts().head(20))
df = df.drop("_merge", axis=1)

## 7. Codes ending in zeros: score of the known group
When the answer is too vague, the SHP pads the code with zeros (`2500` = "IT professional, not further specified").
These codes do not exist in the ILO index. They receive the **mean ILO score of the known group** (`2500` → mean of all `25xx`),
and `exposure_source` records where the score comes from (robustness check possible without the `group` rows).

In [ ]:
# Mean ILO scores at the 1-, 2- and 3-digit levels
group_means = []
for k in [1, 2, 3]:
    m = ilo.groupby(ilo["isco4"].str[:k])["exposure_mean_2025"].mean()
    group_means.append(m)
group_means = pd.concat(group_means)   # index = "3", "25", "214", ...

# Known part of the code: trailing zeros removed (2500 -> "25")
df["isco_known"] = df["isco4"].str.rstrip("0")

df["exposure_main"] = df["exposure4"].fillna(df["isco_known"].map(group_means))
df["exposure_source"] = np.where(df["exposure4"].notna(), "isco4", "group")
df.loc[df["exposure_main"].isna(), "exposure_source"] = "none"

print(df["exposure_source"].value_counts())

## 8. Save (private folder, never on GitHub)

In [ ]:
df.to_parquet(OUT_PATH, index=False)
print("Saved:", OUT_PATH, df.shape)

## 9. Summary

- The SHP add-on (wide format, one column per wave) is reshaped to long format and merged with the cleaned file on `idpers` + `year`: all 40,796 rows find a 4-digit code, and the first three digits match `isco3` in 100% of cases.
- 90.4% of rows receive an exact 4-digit ILO score. 9.6% have vague codes padded with zeros (e.g. `2500`, `3000`) and receive the mean score of their known group. 0.2% (80 rows, occupations absent from the ILO index) have no score and are excluded from the analysis.
- New columns: `isco4` (4-digit code, string), `exposure4` (exact ILO score), `exposure_main` (score used in the analysis), `exposure_source` (`isco4` / `group` / `none`, for robustness checks).